# 3. SBI Model Comparison: BE vs SC

| Part | Data | Fitting target | CV protocol | Cost |
|------|------|---------------|-------------|------|
| **1** | Expert only | Broad stats (SNPE) → predict UM | 2-fold × 64, ANOVA | ~1h total |
| **2** | Expert only | UM directly (per-animal SNPE) | 2-fold × 64, ANOVA | ~20 min/animal |
| **3** | All sessions | Broad stats (SNPE) → predict UM | 2-fold × 64, ANOVA | ~1h total |

In [ ]:
%matplotlib inline
import os, sys
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from behav_utils.data.structures import AnimalData, FittingData
from behav_utils.data.loading import load_experiment

from sbi_comparison_utils import (
    select_expert_sessions, select_all_sessions,
    estimate_timing, print_timing_report,
    train_amortised_snpe, train_per_animal_snpe,
    condition_on_animal, cv_um_comparison, compare_models,
    run_animal_pipeline, run_animal_pipeline_part2,
    simulate_example_session, plot_example_session,
    plot_cv_comparison,
)

try:
    import torch; SBI_OK = True
    print(f"torch={torch.__version__}, {'cuda' if torch.cuda.is_available() else 'cpu'}")
except ImportError:
    SBI_OK = False; print("SBI not available")

## 0. Configuration

In [ ]:
CONFIG_PATH = '../config.yaml'

# ── Session selection ─────────────────────────────────────────────────
STAGE = 'Full_Task_Cont'
DISTRIBUTION = 'Uniform'
MIN_VALID_TRIALS = 30
TARGET_ANIMAL = None       # e.g. 'SS05' or None = all

# ── Expert definition ─────────────────────────────────────────────────
EXPERT_MIN_ACCURACY = 0.70
EXPERT_LAST_FRACTION = 0.50

# ── Stats for Parts 1 & 3 (NO update_matrix — sequence-independent) ──
SBI_STATS = [
    'accuracy', 'psychometric', 'psychometric_gof',
    'recency', 'stimulus_recency', 'win_stay', 'lose_shift',
    'stimulus_sensitivity', 'hard_easy_ratio',
    'logistic_history',
]

# ── Stats for Part 2 (WITH update_matrix — uses real stimuli) ────────
SBI_STATS_WITH_UM = SBI_STATS + ['update_matrix']

# ── SNPE training ─────────────────────────────────────────────────────
N_SBI_SIMS = 50_000        # Amortised (Parts 1 & 3)
N_SBI_SIMS_P2 = 10_000    # Per-animal (Part 2) — less needed
N_GENERIC_TRIALS = 2500    # Generic stimulus sequence length
EXPERT_BURN_IN = 1000
FULL_BURN_IN = 0
SEED = 42

# ── Cross-validation ──────────────────────────────────────────────────
N_CV_FOLDS = 2
N_CV_REPEATS = 64

# ── Plotting ──────────────────────────────────────────────────────────
BE_COL = 'steelblue'
SC_COL = 'darkorange'

In [ ]:
experiment = load_experiment(CONFIG_PATH)

animals = []
for a in experiment.animals:
    try:
        fd = select_expert_sessions(a, STAGE, DISTRIBUTION,
                                     EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
        n_exp = fd.n_sessions
    except ValueError:
        n_exp = 0
    n_all = len(a.get_sessions(stage=STAGE, distribution=DISTRIBUTION))
    ok = n_exp >= 5
    print(f"  {a.animal_id}: {n_all} total, {n_exp} expert {'  ok' if ok else '  skip'}")
    if ok and (TARGET_ANIMAL is None or a.animal_id == TARGET_ANIMAL):
        animals.append(a)

print(f"\nFitting: {[a.animal_id for a in animals]}")

## 0b. Timing estimation

**Run this before committing.** Shows per-simulation cost and projected
total time for each part.

In [ ]:
# Part 1 & 3: amortised stats (no UM)
t_amort = estimate_timing(
    SBI_STATS, n_trials=N_GENERIC_TRIALS,
    burn_in=EXPERT_BURN_IN, n_sbi_sims=N_SBI_SIMS,
)
print_timing_report(t_amort, N_SBI_SIMS, label='Parts 1 & 3 (amortised, no UM)')

# Part 2: per-animal with UM
t_p2 = estimate_timing(
    SBI_STATS_WITH_UM, n_trials=N_GENERIC_TRIALS,
    burn_in=EXPERT_BURN_IN, n_sbi_sims=N_SBI_SIMS_P2,
)
print_timing_report(t_p2, N_SBI_SIMS_P2, n_animals=len(animals),
                    label='Part 2 (per-animal, with UM)')

# Full trajectory (burn_in=0)
t_full = estimate_timing(
    SBI_STATS, n_trials=5000,
    burn_in=FULL_BURN_IN, n_sbi_sims=N_SBI_SIMS,
)
print_timing_report(t_full, N_SBI_SIMS, label='Part 3 (full trajectory, no UM)')

# Total estimate
p1_h = t_amort['be']['total_hours'] + t_amort['sc']['total_hours']
p2_h = (t_p2['be']['total_hours'] + t_p2['sc']['total_hours']) * len(animals)
p3_h = t_full['be']['total_hours'] + t_full['sc']['total_hours']
print(f"\n{'='*60}")
print(f"  TOTAL ESTIMATED TIME")
print(f"  Part 1 (train once):       {p1_h:.1f}h")
print(f"  Part 2 ({len(animals)} animals):     {p2_h:.1f}h")
print(f"  Part 3 (train once):       {p3_h:.1f}h")
print(f"  CV loops (all parts):      ~0.5h")
print(f"  ─────────────────────────────")
print(f"  Grand total:               ~{p1_h + p2_h + p3_h + 0.5:.1f}h")
print(f"{'='*60}")
print(f"\nIf Part 2 is too slow, reduce N_SBI_SIMS_P2 or drop UM from stats.")
print(f"If everything is too slow, use SBI_STATS = ['accuracy', 'psychometric',")
print(f"    'recency', 'win_stay', 'stimulus_sensitivity'] (8 dims, ~10x faster).")

---
# Part 1: Expert — Our Approach

Amortised SNPE on broad stats → predict UM on held-out folds.

In [ ]:
# 1.1 Train amortised SNPE (once for all animals)
if SBI_OK:
    be_snpe_exp = train_amortised_snpe(
        'be', SBI_STATS, N_SBI_SIMS, N_GENERIC_TRIALS, EXPERT_BURN_IN, SEED)
    sc_snpe_exp = train_amortised_snpe(
        'sc', SBI_STATS, N_SBI_SIMS, N_GENERIC_TRIALS, EXPERT_BURN_IN, SEED+1)

In [ ]:
# 1.2 Per-animal: condition + CV + example session
p1 = []
if SBI_OK:
    for animal in animals:
        expert_fd = select_expert_sessions(
            animal, STAGE, DISTRIBUTION, EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
        r = run_animal_pipeline(
            animal, expert_fd, be_snpe_exp, sc_snpe_exp,
            n_cv_repeats=N_CV_REPEATS, seed=SEED)
        p1.append(r)

        plot_cv_comparison(r['be_cv'], r['sc_cv'],
            compare_models(r['be_cv'], r['sc_cv']),
            animal.animal_id, '(Part 1: our approach)')

        ex = simulate_example_session(
            animal, -3, r['be_params'], r['sc_params'],
            STAGE, DISTRIBUTION, EXPERT_BURN_IN)
        plot_example_session(ex, animal.animal_id)

In [ ]:
# 1.3 Summary
if p1:
    df1 = pd.DataFrame([{k: r[k] for k in
        ['animal_id','n_sessions','n_trials','winner','p','be_mean','sc_mean']
    } for r in p1])
    print("Part 1 — Expert, Our Approach:")
    print(df1.to_string(index=False))
    print(f"Tally: {df1['winner'].value_counts().to_dict()}")

---
# Part 2: Expert — Manuscript Approach

Per-animal SNPE with `update_matrix` in the observation vector.
Uses each animal's real stimulus sequence — no sequence mismatch.
Same CV protocol as the manuscript (2-fold × 64, ANOVA on UM MSE).

The difference from grid search: SNPE finds parameters more efficiently,
but the fitting target (reproduce the UM) and evaluation protocol are identical.

In [ ]:
# 2.1 Per-animal: train SNPE with UM + CV
p2 = []
for animal in animals:
    expert_fd = select_expert_sessions(
        animal, STAGE, DISTRIBUTION, EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
    r = run_animal_pipeline_part2(
        animal, expert_fd, SBI_STATS_WITH_UM,
        n_sbi_sims=N_SBI_SIMS_P2, n_cv_repeats=N_CV_REPEATS,
        burn_in=EXPERT_BURN_IN, seed=SEED)
    p2.append(r)

    plot_cv_comparison(r['be_cv'], r['sc_cv'],
        compare_models(r['be_cv'], r['sc_cv']),
        animal.animal_id, '(Part 2: manuscript approach)')

In [ ]:
# 2.2 Summary + agreement with Part 1
if p2:
    df2 = pd.DataFrame([{k: r[k] for k in
        ['animal_id','winner','p','be_mean','sc_mean']
    } for r in p2])
    print("Part 2 — Expert, Manuscript Approach:")
    print(df2.to_string(index=False))
    print(f"Tally: {df2['winner'].value_counts().to_dict()}")

    if p1:
        agree = pd.DataFrame({
            'animal': [r['animal_id'] for r in p1],
            'Part1': [r['winner'] for r in p1],
            'Part2': [r['winner'] for r in p2],
        })
        agree['match'] = agree['Part1'] == agree['Part2']
        print(f"\nAgreement: {agree['match'].mean():.0%}")
        print(agree.to_string(index=False))

---
# Part 3: Full Trajectory — Our Approach

All Uniform sessions (including learning). Burn-in = 0.

In [ ]:
# 3.1 Train amortised SNPE (once)
if SBI_OK:
    be_snpe_full = train_amortised_snpe(
        'be', SBI_STATS, N_SBI_SIMS, 5000, FULL_BURN_IN, SEED+10)
    sc_snpe_full = train_amortised_snpe(
        'sc', SBI_STATS, N_SBI_SIMS, 5000, FULL_BURN_IN, SEED+11)

In [ ]:
# 3.2 Per-animal
p3 = []
if SBI_OK:
    for animal in animals:
        full_fd = select_all_sessions(animal, STAGE, DISTRIBUTION, MIN_VALID_TRIALS)
        r = run_animal_pipeline(
            animal, full_fd, be_snpe_full, sc_snpe_full,
            n_cv_repeats=N_CV_REPEATS, seed=SEED)
        p3.append(r)

        plot_cv_comparison(r['be_cv'], r['sc_cv'],
            compare_models(r['be_cv'], r['sc_cv']),
            animal.animal_id, '(Part 3: full trajectory)')

In [ ]:
if p3:
    df3 = pd.DataFrame([{k: r[k] for k in
        ['animal_id','n_sessions','n_trials','winner','p','be_mean','sc_mean']
    } for r in p3])
    print("Part 3 — Full Trajectory:")
    print(df3.to_string(index=False))
    print(f"Tally: {df3['winner'].value_counts().to_dict()}")

---
# Grand Summary

In [ ]:
if p1 and p2 and p3:
    grand = pd.DataFrame({
        'animal': [r['animal_id'] for r in p1],
        'P1_expert_ours': [r['winner'] for r in p1],
        'P2_expert_manuscript': [r['winner'] for r in p2],
        'P3_full_ours': [r['winner'] for r in p3],
    })
    print("Model assignment per animal:\n")
    print(grand.to_string(index=False))

    print("\nTallies:")
    for col in grand.columns[1:]:
        print(f"  {col}: {grand[col].value_counts().to_dict()}")

    print(f"\nP1 vs P2 agreement: {(grand.iloc[:,1]==grand.iloc[:,2]).mean():.0%}")
    print(f"P1 vs P3 agreement: {(grand.iloc[:,1]==grand.iloc[:,3]).mean():.0%}")
    print(f"P2 vs P3 agreement: {(grand.iloc[:,2]==grand.iloc[:,3]).mean():.0%}")

    # Scatter: Part 1 vs Part 2 error ratios
    fig, ax = plt.subplots(figsize=(6, 6))
    p1_ratio = np.array([r['be_mean'] for r in p1]) / np.array([r['sc_mean'] for r in p1])
    p2_ratio = np.array([r['be_mean'] for r in p2]) / np.array([r['sc_mean'] for r in p2])
    ax.scatter(p1_ratio, p2_ratio, s=60, alpha=0.7)
    for i, r in enumerate(p1):
        ax.annotate(r['animal_id'], (p1_ratio[i], p2_ratio[i]), fontsize=8)
    ax.axhline(1, color='grey', ls='--', alpha=0.3)
    ax.axvline(1, color='grey', ls='--', alpha=0.3)
    ax.set_xlabel('Part 1: BE/SC error ratio')
    ax.set_ylabel('Part 2: BE/SC error ratio')
    ax.set_title('Method agreement (< 1 = BE wins)')
    plt.tight_layout(); plt.show()

---
# Session-by-Session Visualisation

For each animal, shows:
- **UM grid**: rows = sessions, columns = Real | BE | SC
- **Psychometric grid**: Real (black) + BE (blue) + SC (orange) overlaid
- **Pooled UM comparison**: averaged across all sessions

Uses Part 1 (our approach) params by default. Change `results_to_use`
to `p2` for manuscript-approach params.

In [ ]:
from sbi_comparison_utils import (
    simulate_all_sessions,
    plot_session_by_session_um,
    plot_session_by_session_psychometric,
    plot_pooled_um_comparison,
)

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
results_to_use = p1        # Change to p2 for manuscript params
VIS_BURN_IN = EXPERT_BURN_IN
VIS_N_REPS = 20            # Stochastic realisations per model per session
VIS_MAX_SESSIONS = 20      # Cap for very long session lists

In [ ]:
# Generate session-by-session data for each animal
all_session_data = {}

for r in results_to_use:
    aid = r['animal_id']
    animal = experiment.get_animal(aid)
    print(f"\nSimulating all sessions for {aid}...")

    sdata = simulate_all_sessions(
        animal,
        be_params=r['be_params'],
        sc_params=r['sc_params'],
        stage=STAGE, distribution=DISTRIBUTION,
        min_accuracy=EXPERT_MIN_ACCURACY,
        last_fraction=EXPERT_LAST_FRACTION,
        burn_in=VIS_BURN_IN,
        n_reps=VIS_N_REPS,
        seed=SEED,
    )
    all_session_data[aid] = sdata
    print(f"  {len(sdata)} sessions simulated")

In [ ]:
# Plot for each animal
for aid, sdata in all_session_data.items():
    print(f"\n{'='*60}")
    print(f"  {aid}")
    print(f"{'='*60}")

    # Pooled UM comparison (most important — manuscript-style figure)
    plot_pooled_um_comparison(sdata, aid)

    # Psychometric curves (compact grid)
    plot_session_by_session_psychometric(
        sdata, aid, max_sessions=VIS_MAX_SESSIONS)

    # Full UM grid (large figure — scroll down)
    plot_session_by_session_um(
        sdata, aid, max_sessions=VIS_MAX_SESSIONS)

In [ ]:
# Per-session MSE trajectory: how does model fit evolve across sessions?
for aid, sdata in all_session_data.items():
    if not sdata:
        continue

    sess_idx = [d['session_idx'] for d in sdata]
    be_mses = [d['be_um_mse'] for d in sdata]
    sc_mses = [d['sc_um_mse'] for d in sdata]
    accs = [d['accuracy'] for d in sdata]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax = axes[0]
    ax.plot(sess_idx, be_mses, 'o-', color=BE_COL, label='BE', markersize=5)
    ax.plot(sess_idx, sc_mses, 's-', color=SC_COL, label='SC', markersize=5)
    ax.set_ylabel('UM MSE')
    ax.set_title(f'{aid} — Per-session UM MSE')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)

    ax = axes[1]
    ax.plot(sess_idx, accs, 'ko-', markersize=5)
    ax.axhline(EXPERT_MIN_ACCURACY, color='red', ls='--', alpha=0.4,
               label=f'Expert threshold ({EXPERT_MIN_ACCURACY})')
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Session index')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)

    plt.tight_layout()
    plt.show()